# Setup and Installations
Install required dependencies.

In [ ]:
!pip install pytorch-lightning albumentations torchmetrics transformers timm segmentation-models-pytorch scikit-learn kagglehub

# Kaggle Dataset Download
Configure Kaggle credentials and download the ISIC 2018 Task 1 dataset.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("tschandl/isic2018-challenge-task1-data-segmentation")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'isic2018-challenge-task1-data-segmentation' dataset.
Path to dataset files: /kaggle/input/isic2018-challenge-task1-data-segmentation


# Imports

In [ ]:
import os
import glob
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True


In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Root directory on Drive where all checkpoints and logs will be persisted.
# Change this path if you prefer a different folder name.
DRIVE_CKPT_DIR = "/content/drive/MyDrive/ISIC2018_checkpoints"
os.makedirs(f"{DRIVE_CKPT_DIR}/unet",      exist_ok=True)
os.makedirs(f"{DRIVE_CKPT_DIR}/transunet", exist_ok=True)
print(f"Checkpoint root: {DRIVE_CKPT_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint root: /content/drive/MyDrive/ISIC2018_checkpoints


# Dataset Definition

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def _load_sample(args):
    img_path, mask_path = args
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)
    mask = (mask > 127).astype(np.float32)
    return img, mask


class ISIC2018Dataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.transform = transform
        # Pre-load and resize all images/masks into CPU RAM to eliminate
        # per-step disk I/O. At 256×256 this costs ~530 MB images + ~180 MB masks.
        # ThreadPoolExecutor parallelises disk reads (I/O releases the GIL),
        # cutting cache time from ~5 min down to ~1 min on Colab.
        print(f"  Caching {len(image_paths)} samples into RAM …", end=" ", flush=True)
        with ThreadPoolExecutor(max_workers=8) as executor:
            results = list(executor.map(_load_sample, zip(image_paths, mask_paths)))
        self.images, self.masks = map(list, zip(*results))
        print("done.")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # .copy() so albumentations never mutates the cached array
        image = self.images[idx].copy()
        mask  = self.masks[idx].copy()

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask  = augmented['mask']

        if not torch.is_tensor(mask):
            mask = torch.tensor(mask)
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)

        return image, mask


# DataModule Definition

In [ ]:
class ISICDataModule(pl.LightningDataModule):
    def __init__(self, data_dir="isic2018-task1", batch_size=16, num_workers=2):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = min(num_workers, os.cpu_count() or 1)
        self._setup_done = False
        # Persistent references to keep data in RAM
        self.train_dataset = None
        self.val_dataset = None

    def setup(self, stage=None):
        # 1. Check if already marked done
        if self._setup_done:
            return

        # 2. Check if datasets are already in RAM to avoid re-caching
        if self.train_dataset is not None and self.val_dataset is not None:
            print("  Data already in RAM. Skipping re-cache.")
            self._setup_done = True
            return

        image_paths = sorted(glob.glob(os.path.join(self.data_dir, "ISIC2018_Task1-2_Training_Input", "*.jpg")))
        mask_paths = sorted(glob.glob(os.path.join(self.data_dir, "ISIC2018_Task1_Training_GroundTruth", "*.png")))

        if len(image_paths) == 0:
            print("Warning: No images found. Check data extraction paths.")

        train_imgs, val_imgs, train_masks, val_masks = train_test_split(
            image_paths, mask_paths, test_size=0.1, random_state=42
        )

        train_transform = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.Affine(translate_percent=0.0625, scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
            A.ColorJitter(p=0.5),
            A.GaussianBlur(p=0.2),
            A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 16), hole_width_range=(1, 16), p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

        val_transform = A.Compose([
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2()
        ])

        self.train_dataset = ISIC2018Dataset(train_imgs, train_masks, transform=train_transform)
        self.val_dataset = ISIC2018Dataset(val_imgs, val_masks, transform=val_transform)
        self._setup_done = True

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True, drop_last=True,
                          persistent_workers=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True,
                          persistent_workers=True)

# Loss Functions and Architectures

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import torchmetrics
import timm

class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super(BCEDiceLoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)

        intersection = (probs_flat * targets_flat).sum()
        dice_score = (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        dice_loss = 1.0 - dice_score

        return bce_loss + dice_loss

# Lightning Module Base: Segmentation System

In [ ]:
class SegmentationSystem(pl.LightningModule):
    def __init__(self, model, lr=1e-4):
        super(SegmentationSystem, self).__init__()
        self.model = model
        self.lr = lr
        self.criterion = BCEDiceLoss()
        self.train_dice = torchmetrics.F1Score(task="binary")
        self.val_dice = torchmetrics.F1Score(task="binary")
        self.train_iou = torchmetrics.JaccardIndex(task="binary")
        self.val_iou = torchmetrics.JaccardIndex(task="binary")

        self.train_recall = torchmetrics.Recall(task="binary")
        self.val_recall = torchmetrics.Recall(task="binary")

        self.train_specificity = torchmetrics.Specificity(task="binary")
        self.val_specificity = torchmetrics.Specificity(task="binary")

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.criterion(logits, masks)

        preds = torch.sigmoid(logits)
        masks_int = masks.int()

        self.train_dice(preds, masks_int)
        self.train_iou(preds, masks_int)
        self.train_recall(preds, masks_int)
        self.train_specificity(preds, masks_int)

        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        self.log("train_dice", self.train_dice, on_epoch=True, prog_bar=True)
        self.log("train_iou", self.train_iou, on_epoch=True)
        self.log("train_recall", self.train_recall, on_epoch=True)
        self.log("train_specificity", self.train_specificity, on_epoch=True)

        return loss

    def validation_step(self, batch, batch_idx):
        images, masks = batch
        logits = self(images)
        loss = self.criterion(logits, masks)

        preds = torch.sigmoid(logits)
        masks_int = masks.int()

        self.val_dice(preds, masks_int)
        self.val_iou(preds, masks_int)
        self.val_recall(preds, masks_int)
        self.val_specificity(preds, masks_int)

        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        self.log("val_dice", self.val_dice, on_epoch=True, prog_bar=True)
        self.log("val_iou", self.val_iou, on_epoch=True)
        self.log("val_recall", self.val_recall, on_epoch=True)
        self.log("val_specificity", self.val_specificity, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        # T_max matches MAX_EPOCHS so the lr decays smoothly across the full run
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=50, eta_min=1e-6
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
            },
        }


# Baseline U-Net Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Encoder
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)

        # Decoder
        for feature in reversed(features):
            self.ups.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.ups.append(DoubleConv(feature*2, feature))

        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]

            if x.shape != skip_connection.shape:
                x = F.interpolate(x, size=skip_connection.shape[2:], mode='bilinear', align_corners=True)

            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](concat_skip)

        return self.final_conv(x)

class UNetSystem(SegmentationSystem):
    def __init__(self, in_channels=3, out_channels=1, lr=1e-4):
        super().__init__(model=UNet(in_channels, out_channels), lr=lr)

# TransUNet Architecture

In [ ]:
class TransUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(TransUNet, self).__init__()

        # ── CNN Encoder (3 levels: 256×256 → 32×32 at 256 channels) ──────────
        self.cnn_encoder = nn.ModuleList([
            DoubleConv(in_channels, 64),
            nn.MaxPool2d(2),
            DoubleConv(64, 128),
            nn.MaxPool2d(2),
            DoubleConv(128, 256),
            nn.MaxPool2d(2),
        ])  # Maps 256×256 → 32×32 feature map at 256 channels

        # ── Channel projection: CNN features (256 ch) → ViT dim (768) ─────────
        # Named `patch_embed` so ProgressiveUnfreeze can target it by attribute.
        self.patch_embed = nn.Conv2d(256, 768, kernel_size=1)

        # ── Pre-trained ViT transformer blocks (ImageNet-21k weights) ──────────
        # We load vit_base_patch16 at img_size=256 (→ 16×16=256 ViT patches).
        # Only the 12 transformer `blocks` are retained; the patch embedding and
        # classification head are discarded since we supply our own CNN features.
        _vit = timm.create_model(
            'vit_base_patch16_224.augreg_in21k',
            pretrained=True,
            img_size=256,
            patch_size=16,
        )
        self.vit_blocks = _vit.blocks  # nn.Sequential of 12 pretrained Blocks

        # Adapt positional embeddings: ViT has (1, 257, 768) [CLS + 16×16 grid].
        # Our CNN tokens are 32×32 = 1 024, so we interpolate the spatial part.
        with torch.no_grad():
            pe = _vit.pos_embed[:, 1:, :]                        # (1, 256, 768)  drop CLS
            pe = pe.reshape(1, 16, 16, 768).permute(0, 3, 1, 2)  # (1, 768, 16, 16)
            pe = F.interpolate(pe, size=(32, 32), mode='bilinear', align_corners=False)
            pe = pe.permute(0, 2, 3, 1).reshape(1, 1024, 768)    # (1, 1024, 768)
        # Register as a non-trainable buffer (moves to device automatically)
        self.pos_embed = nn.Parameter(pe)
        del _vit  # release unused parts (head, patch_embed, cls_token, etc.)

        # ── Bottleneck: reshape 768-dim tokens back to 512-channel spatial map ─
        self.bottleneck_conv = nn.Conv2d(768, 512, kernel_size=3, padding=1)

        # ── U-Net Style Decoder ───────────────────────────────────────────────
        self.up1  = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(512, 256)

        self.up2  = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)

        self.up3  = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # ── CNN encoder with skip connections ─────────────────────────────────
        skip_connections = []
        cnn_in = x
        for i in range(0, len(self.cnn_encoder), 2):
            cnn_in = self.cnn_encoder[i](cnn_in)
            skip_connections.append(cnn_in)
            cnn_in = self.cnn_encoder[i + 1](cnn_in)

        # ── Project CNN features → ViT token sequence ────────────────────────
        B, C, H, W = cnn_in.shape                               # (B, 256, 32, 32)
        x_seq = self.patch_embed(cnn_in)                        # (B, 768, 32, 32)
        x_seq = x_seq.flatten(2).transpose(1, 2)               # (B, 1024, 768)

        # ── Add positional embedding and run pre-trained ViT blocks ───────────
        x_seq = x_seq + self.pos_embed                          # broadcast over batch
        for blk in self.vit_blocks:
            x_seq = blk(x_seq)

        # ── Reshape to spatial map and project to 512 channels ───────────────
        x_spatial   = x_seq.transpose(1, 2).view(B, 768, H, W) # (B, 768, 32, 32)
        x_bottleneck = self.bottleneck_conv(x_spatial)          # (B, 512, 32, 32)

        # ── U-Net decoder with CNN skip connections ───────────────────────────
        x_up = self.up1(x_bottleneck)
        x_up = torch.cat([x_up, skip_connections[-1]], dim=1)
        x_up = self.dec1(x_up)

        x_up = self.up2(x_up)
        x_up = torch.cat([x_up, skip_connections[-2]], dim=1)
        x_up = self.dec2(x_up)

        x_up = self.up3(x_up)
        x_up = torch.cat([x_up, skip_connections[-3]], dim=1)
        x_up = self.dec3(x_up)

        return self.final_conv(x_up)


class TransUNetSystem(SegmentationSystem):
    def __init__(self, in_channels=3, out_channels=1, lr=1e-4):
        super().__init__(model=TransUNet(in_channels, out_channels), lr=lr)

    def configure_optimizers(self):
      # This ensures Phase 1 ONLY trains the decoder/bottleneck.
      # ProgressiveUnfreeze will add the backbone parameters later at Epoch 5.
      trainable_params = filter(lambda p: p.requires_grad, self.parameters())
      optimizer = torch.optim.Adam(trainable_params, lr=self.lr)
      return optimizer

# Progressive Unfreezing Callback
Implements `BaseFinetuning` for a two-phase training schedule:
- **Phase 1 (Epochs 0–9)**: Only the TransUNet bottleneck and decoder train; `cnn_encoder` and `patch_embed` are frozen.
- **Phase 2 (Epoch 10+)**: All parameters unfreeze and train end-to-end.

In [ ]:
from pytorch_lightning.callbacks import BaseFinetuning


class ProgressiveUnfreeze(BaseFinetuning):
    """
    Two-phase progressive unfreezing for TransUNet:

    Phase 1 (Epochs 0 → unfreeze_epoch-1):
        Freeze cnn_encoder, patch_embed, and vit_blocks.
        Gradients flow only through bottleneck_conv and the U-Net decoder.

    Phase 2 (unfreeze_epoch onwards):
        Unfreeze all three modules with a low backbone_lr (default 1e-5) so
        the pretrained ViT weights are adapted gently without being overwritten.
        The decoder param group keeps its original lr (1e-4).
    """

    def __init__(self, unfreeze_epoch: int = 5, backbone_lr: float = 1e-5):
        super().__init__()
        self.unfreeze_epoch = unfreeze_epoch
        self.backbone_lr = backbone_lr

    # Called once before training starts ─────────────────────────────────────
    def freeze_before_training(self, pl_module: pl.LightningModule) -> None:
        if not isinstance(pl_module.model, TransUNet):
            return
        # Phase 1: freeze CNN encoder, projection layer, and ViT transformer
        self.freeze(pl_module.model.cnn_encoder)
        self.freeze(pl_module.model.patch_embed)
        self.freeze(pl_module.model.vit_blocks)

    # Called at the start of every epoch ─────────────────────────────────────
    def finetune_function(
        self,
        pl_module: pl.LightningModule,
        current_epoch: int,
        optimizer: torch.optim.Optimizer,
    ) -> None:
        if not isinstance(pl_module.model, TransUNet):
            return
        if current_epoch == self.unfreeze_epoch:
            # Phase 2: unfreeze with a reduced lr so ViT weights are not
            # overwritten — each backbone group gets backbone_lr (1e-5),
            # while the existing decoder group keeps its original lr (1e-4).
            # Grouping them ensures they share one optimizer update group
            self.unfreeze_and_add_param_group(
                modules=[pl_module.model.cnn_encoder,
                        pl_module.model.patch_embed,
                        pl_module.model.vit_blocks],
                optimizer=optimizer,
                lr=self.backbone_lr,
                train_bn=True,
            )

            print(
                f"[ProgressiveUnfreeze] Epoch {current_epoch}: "
                f"cnn_encoder, patch_embed, vit_blocks unfrozen "
                f"(backbone_lr={self.backbone_lr}) – "
                "full end-to-end fine-tuning begins."
            )


# Trainer Configuration
Builds two `pl.Trainer` instances — one for the U-Net baseline and one for TransUNet — sharing the same 50-epoch budget.
- **TensorBoardLogger** writes scalars to `logs/` for side-by-side comparison in TensorBoard.
- **ModelCheckpoint** saves the single best checkpoint per architecture (by `val_dice`).
- **ProgressiveUnfreeze** unfreezes the TransUNet backbone at epoch 5 with `backbone_lr=1e-5` (10× lower than the decoder) to avoid overwriting pretrained ViT weights.
- **Mixed precision** (`precision="16-mixed"`) halves memory usage and speeds up GPU math.


In [ ]:
from pytorch_lightning.loggers import TensorBoardLogger, CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint

MAX_EPOCHS = 50

# ── Loggers ───────────────────────────────────────────────────────────────────
unet_tb_logger  = TensorBoardLogger(save_dir="logs/", name="unet_baseline")
unet_csv_logger = CSVLogger(save_dir="logs/", name="unet_baseline")

transunet_tb_logger  = TensorBoardLogger(save_dir="logs/", name="transunet")
transunet_csv_logger = CSVLogger(save_dir="logs/", name="transunet")

# ── Callbacks ─────────────────────────────────────────────────────────────────
# dirpath points to Google Drive so checkpoints survive runtime disconnects.
# save_last=True writes a `last.ckpt` that is used to resume training.
unet_ckpt_cb = ModelCheckpoint(
    dirpath=f"{DRIVE_CKPT_DIR}/unet",
    monitor="val_dice",
    mode="max",
    save_top_k=1,
    save_last=True,
    filename="unet-{epoch:02d}-{val_dice:.4f}",
    verbose=True,
)

# unfreeze_epoch=5  → Phase 2 gets 45 epochs instead of 20
# backbone_lr=1e-5  → gentle lr for pretrained ViT/CNN so weights aren't overwritten
progressive_unfreeze_cb = ProgressiveUnfreeze(unfreeze_epoch=5, backbone_lr=1e-5)

transunet_ckpt_cb = ModelCheckpoint(
    dirpath=f"{DRIVE_CKPT_DIR}/transunet",
    monitor="val_dice",
    mode="max",
    save_top_k=1,
    save_last=True,
    filename="transunet-{epoch:02d}-{val_dice:.4f}",
    verbose=True,
)

# ── U-Net Trainer ─────────────────────────────────────────────────────────────
unet_trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    logger=[unet_tb_logger, unet_csv_logger],
    callbacks=[unet_ckpt_cb],
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    log_every_n_steps=10,
)

# ── TransUNet Trainer (with ProgressiveUnfreeze) ──────────────────────────────
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    logger=[transunet_tb_logger, transunet_csv_logger],
    callbacks=[progressive_unfreeze_cb, transunet_ckpt_cb],
    accelerator="auto",
    devices=1,
    precision="16-mixed",
    log_every_n_steps=10,
)


INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts a

# Model Training Execution
Runs the comparison ablation sequentially:
1. **U-Net baseline** trained for 30 epochs with `unet_trainer`.
2. GPU cache cleared via `torch.cuda.empty_cache()`.
3. **TransUNet** trained for 30 epochs with `trainer` (includes `ProgressiveUnfreeze`).


In [ ]:
import shutil

# ── Fresh-start flag ──────────────────────────────────────────────────────────
# Set to True when you change hyperparameters (epochs, lr, unfreeze schedule)
# so the old Drive checkpoints are not silently resumed with wrong settings.
# Set to False to resume from the last saved checkpoint after a disconnect.
FRESH_START = False

# Set to True to clear only TransUNet checkpoints and restart its training.
# This flag takes precedence over FRESH_START for TransUNet if both are True.
RESTART_TRANSUNET = False

if FRESH_START:
    # If FRESH_START is True, clear both U-Net and TransUNet checkpoints
    for subdir in ["unet", "transunet"]:
        ckpt_dir = f"{DRIVE_CKPT_DIR}/{subdir}"
        if os.path.isdir(ckpt_dir):
            shutil.rmtree(ckpt_dir)
            os.makedirs(ckpt_dir, exist_ok=True)
            print(f"  Cleared checkpoint dir: {ckpt_dir}")
elif RESTART_TRANSUNET:
    # If RESTART_TRANSUNET is True (and FRESH_START is False), clear only TransUNet checkpoints
    transunet_ckpt_dir = f"{DRIVE_CKPT_DIR}/transunet"
    if os.path.isdir(transunet_ckpt_dir):
        shutil.rmtree(transunet_ckpt_dir)
        os.makedirs(transunet_ckpt_dir, exist_ok=True)
        print(f"  Cleared TransUNet checkpoint dir: {transunet_ckpt_dir}")

# ── Shared DataModule ─────────────────────────────────────────────────────────
data_module = ISICDataModule(data_dir=path, batch_size=16, num_workers=4)

# ── Resume helpers ────────────────────────────────────────────────────────────
def _last_ckpt(dirpath: str) -> str | None:
    """Return the path to last.ckpt if it exists in dirpath, else None."""
    p = os.path.join(dirpath, "last.ckpt")
    return p if os.path.isfile(p) else None

unet_resume      = _last_ckpt(f"{DRIVE_CKPT_DIR}/unet")
transunet_resume = _last_ckpt(f"{DRIVE_CKPT_DIR}/transunet")

# ══════════════════════════════════════════════════════════════════════════════
# 1.  U-Net Baseline
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  Training U-Net Baseline  (50 epochs)")
if unet_resume:
    print(f"  Resuming from: {unet_resume}")
print("=" * 60)

unet_system = UNetSystem(in_channels=3, out_channels=1, lr=1e-4)
unet_trainer.fit(unet_system, datamodule=data_module, ckpt_path=unet_resume)

# Explicitly move the U-Net model to CPU and clear GPU cache
unet_system.to("cpu")
del unet_system
torch.cuda.empty_cache()
print("\n--- GPU Memory after U-Net clearing ---")
print(torch.cuda.memory_summary())
print("-------------------------------------\n")

# ══════════════════════════════════════════════════════════════════════════════
# 2.  TransUNet  (with ProgressiveUnfreeze)
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 60)
print("  Training TransUNet  (50 epochs, progressive unfreezing at epoch 5)")
if transunet_resume:
    print(f"  Resuming from: {transunet_resume}")
print("=" * 60)

transunet_system = TransUNetSystem(in_channels=3, out_channels=1, lr=1e-4)
trainer.fit(transunet_system, datamodule=data_module, ckpt_path=transunet_resume)

torch.cuda.empty_cache()
print("\n--- GPU Memory after TransUNet clearing ---")
print(torch.cuda.memory_summary())
print("-------------------------------------------\n")

Epoch 23/49 ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/145 0:00:24 • 0:03:09 0.75it/s v_num: 0_1 train_loss_step: 0.199  
                                                                                train_dice_step: 0.911             

# Validation / Inference Visualization
Loads the held-out validation partition and renders a 4-column grid per sample:

| Input Image | Ground Truth | U-Net Prediction | TransUNet Prediction |
|-------------|-------------|-----------------|---------------------|

Logits are converted to probabilities via **Sigmoid**, then binarised at threshold **0.5**.
All tensors are moved to CPU before `.detach().numpy()` to satisfy NumPy's requirement.

In [ ]:
import matplotlib.pyplot as plt


def visualize_predictions(
    unet_sys: pl.LightningModule,
    transunet_sys: pl.LightningModule,
    datamodule: pl.LightningDataModule,
    num_samples: int = 4,
    threshold: float = 0.5,
) -> None:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    unet_sys.eval().to(device)
    transunet_sys.eval().to(device)

    # datamodule.setup() is intentionally NOT called here — the setup guard in
    # ISICDataModule ensures it only caches once; calling it again would be a no-op
    # but the explicit call was previously causing spurious RAM spikes.
    val_loader = datamodule.val_dataloader()

    imgs_buf, masks_buf, unet_buf, trans_buf = [], [], [], []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)

            unet_prob = torch.sigmoid(unet_sys(images)).cpu()
            trans_prob = torch.sigmoid(transunet_sys(images)).cpu()

            unet_bin = (unet_prob >= threshold).float()
            trans_bin = (trans_prob >= threshold).float()

            imgs_buf.append(images.cpu())
            masks_buf.append(masks.cpu())
            unet_buf.append(unet_bin)
            trans_buf.append(trans_bin)

            collected = sum(b.shape[0] for b in imgs_buf)
            if collected >= num_samples:
                break

    imgs = torch.cat(imgs_buf, dim=0)[:num_samples]
    masks = torch.cat(masks_buf, dim=0)[:num_samples]
    unet_preds = torch.cat(unet_buf, dim=0)[:num_samples]
    trans_preds = torch.cat(trans_buf, dim=0)[:num_samples]

    col_titles = ["Input Image", "Ground Truth", "U-Net Prediction", "TransUNet Prediction"]
    fig, axes = plt.subplots(num_samples, 4, figsize=(18, num_samples * 4.5))

    if num_samples == 1:
        axes = axes[None, :]

    for col_ax, title in zip(axes[0], col_titles):
        col_ax.set_title(title, fontsize=13, fontweight="bold")

    for i in range(num_samples):
        img_np = imgs[i].permute(1, 2, 0).detach().numpy()
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

        gt_np = masks[i].squeeze().detach().numpy()
        unet_np = unet_preds[i].squeeze().detach().numpy()
        trans_np = trans_preds[i].squeeze().detach().numpy()

        axes[i, 0].imshow(img_np);        axes[i, 0].axis("off")
        axes[i, 1].imshow(gt_np,    cmap="gray", vmin=0, vmax=1); axes[i, 1].axis("off")
        axes[i, 2].imshow(unet_np,  cmap="gray", vmin=0, vmax=1); axes[i, 2].axis("off")
        axes[i, 3].imshow(trans_np, cmap="gray", vmin=0, vmax=1); axes[i, 3].axis("off")

    fig.suptitle(
        "Segmentation Comparison — Ground Truth vs. Model Predictions",
        fontsize=15, y=1.01,
    )
    plt.tight_layout()
    plt.show()


# ── Run visualisation on the trained models ───────────────────────────────────
# Reload the models from their best checkpoints for visualization
unet_system = UNetSystem.load_from_checkpoint(unet_ckpt_cb.best_model_path, in_channels=3, out_channels=1, lr=1e-4)
transunet_system = TransUNetSystem.load_from_checkpoint(transunet_ckpt_cb.best_model_path, in_channels=3, out_channels=1, lr=1e-4)

visualize_predictions(
    unet_sys=unet_system,
    transunet_sys=transunet_system,
    datamodule=data_module,
    num_samples=4,
    threshold=0.5,
)

# Clear memory after visualization
del unet_system
del transunet_system
torch.cuda.empty_cache()

# Ablation Study — Test-Set Evaluation & Results Table
Loads the best saved checkpoint for each architecture and runs a final validation pass to collect the four segmentation metrics. The results are assembled into a three-row markdown-style comparison table:

| Architecture | Dice ↑ | IoU ↑ | Recall ↑ | Specificity ↑ |
|---|---|---|---|---|
| U-Net (scratch) | … | … | … | … |
| TransUNet — Phase 1: frozen backbone (epoch 0–9) | … | … | … | … |
| TransUNet — Phase 2: full fine-tune (best ckpt) | … | … | … | … |

Phase 1 metrics are extracted from the CSVLogger output file at epoch 9.
Phase 2 metrics are obtained by running `trainer.validate()` on the best checkpoint.

In [ ]:
import os
import glob
import pandas as pd


# ── Helpers ───────────────────────────────────────────────────────────────────

def load_latest_csv(log_name: str) -> pd.DataFrame:
    pattern = os.path.join("logs", log_name, "version_*", "metrics.csv")
    csv_files = sorted(glob.glob(pattern))
    if not csv_files:
        raise FileNotFoundError(
            f"No metrics.csv found for '{log_name}'. "
            "Ensure training has completed and CSVLogger is active."
        )
    return pd.read_csv(csv_files[-1])


def epoch_val_metrics(df: pd.DataFrame, epoch: int) -> dict:
    keys = ["val_dice", "val_iou", "val_recall", "val_specificity"]
    val_rows = df.dropna(subset=["val_dice"])
    epoch_row = val_rows[val_rows["epoch"] == epoch]
    if epoch_row.empty:
        closest = (val_rows["epoch"] - epoch).abs().idxmin()
        epoch_row = val_rows.loc[[closest]]
    row = epoch_row.iloc[-1]
    return {k: float(row.get(k, float("nan"))) for k in keys}


def best_val_metrics(df: pd.DataFrame) -> dict:
    val_rows = df.dropna(subset=["val_dice"])
    best_idx = val_rows["val_dice"].idxmax()
    row = val_rows.loc[best_idx]
    keys = ["val_dice", "val_iou", "val_recall", "val_specificity"]
    return {k: float(row.get(k, float("nan"))) for k in keys}


# ── Load checkpoint metrics via trainer.validate() ────────────────────────────
# data_module.setup() is NOT called here — the setup guard in ISICDataModule
# ensures images are only cached once. Lightning will call setup() internally
# via trainer.validate() if needed.

# Best U-Net checkpoint
print("Evaluating U-Net best checkpoint …")
unet_best = UNetSystem.load_from_checkpoint(
    unet_ckpt_cb.best_model_path,
    in_channels=3, out_channels=1, lr=1e-4,
)
unet_val_results = unet_trainer.validate(unet_best, datamodule=data_module, verbose=False)
unet_best_metrics = unet_val_results[0]

# Best TransUNet checkpoint (full fine-tune, Phase 2)
print("Evaluating TransUNet best checkpoint (full fine-tune) …")
transunet_best = TransUNetSystem.load_from_checkpoint(
    transunet_ckpt_cb.best_model_path,
    in_channels=3, out_channels=1, lr=1e-4,
)
transunet_val_results = trainer.validate(transunet_best, datamodule=data_module, verbose=False)
transunet_best_metrics = transunet_val_results[0]

torch.cuda.empty_cache()

# ── Phase 1 metrics from CSVLogger (epoch 9 = last frozen epoch) ─────────────
print("Reading Phase 1 (frozen backbone) metrics from CSVLogger …")
try:
    transunet_csv = load_latest_csv("transunet")
    phase1_metrics = epoch_val_metrics(transunet_csv, epoch=9)
except FileNotFoundError as exc:
    print(f"  Warning: {exc}")
    phase1_metrics = {k: float("nan") for k in
                      ["val_dice", "val_iou", "val_recall", "val_specificity"]}

# ── Assemble ablation table ───────────────────────────────────────────────────

def fmt(val) -> str:
    return f"{val:.4f}" if not pd.isna(val) else "  N/A "

rows = [
    {
        "Architecture":  "U-Net (scratch)",
        "Dice":          fmt(unet_best_metrics.get("val_dice",        float("nan"))),
        "IoU":           fmt(unet_best_metrics.get("val_iou",         float("nan"))),
        "Recall":        fmt(unet_best_metrics.get("val_recall",      float("nan"))),
        "Specificity":   fmt(unet_best_metrics.get("val_specificity", float("nan"))),
    },
    {
        "Architecture":  "TransUNet — Phase 1: frozen backbone (epochs 0-9)",
        "Dice":          fmt(phase1_metrics["val_dice"]),
        "IoU":           fmt(phase1_metrics["val_iou"]),
        "Recall":        fmt(phase1_metrics["val_recall"]),
        "Specificity":   fmt(phase1_metrics["val_specificity"]),
    },
    {
        "Architecture":  "TransUNet — Phase 2: full fine-tune (best ckpt)",
        "Dice":          fmt(transunet_best_metrics.get("val_dice",        float("nan"))),
        "IoU":           fmt(transunet_best_metrics.get("val_iou",         float("nan"))),
        "Recall":        fmt(transunet_best_metrics.get("val_recall",      float("nan"))),
        "Specificity":   fmt(transunet_best_metrics.get("val_specificity", float("nan"))),
    },
]

col_w = {
    "Architecture": max(len(r["Architecture"]) for r in rows),
    "Dice": 10, "IoU": 10, "Recall": 10, "Specificity": 13,
}

def hline():
    return "+" + "+".join("-" * (col_w[c] + 2) for c in col_w) + "+"

def row_str(r):
    cells = [f" {r[c]:<{col_w[c]}} " for c in col_w]
    return "|" + "|".join(cells) + "|"

header = {c: c for c in col_w}

print()
print(hline())
print(row_str(header))
print(hline())
for r in rows:
    print(row_str(r))
print(hline())
print()
print("Tip: launch TensorBoard with  `%load_ext tensorboard && %tensorboard --logdir logs/`")
print("     to view per-epoch training curves for both architectures side by side.")


## URL Image Inference — Overlay Predictions on Any Image
Set `IMAGE_URL` to any publicly accessible dermoscopy (or other RGB) image.
The cell downloads the image, runs both trained models, and renders the segmentation masks blended over the original.

In [ ]:
import io
import urllib.request
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os # Import os module to check file existence

# ── Config ────────────────────────────────────────────────────────────────────
IMAGE_URL  = "https://www.minarsdermatology.com/wp-content/uploads/2025/06/abcdes-of-melanoma-signs-of-skin-cancer.jpg"  # <- replace with your URL
THRESHOLD  = 0.5      # binarisation threshold for predicted mask
ALPHA      = 0.45     # overlay opacity  (0 = invisible, 1 = opaque)
UNET_COLOR      = (255,  50,  50)  # red  — U-Net overlay
TRANSUNET_COLOR = ( 50, 200, 255)  # cyan — TransUNet overlay

# ── 1. Download image ─────────────────────────────────────────────────────────
req = urllib.request.Request(IMAGE_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=15) as resp:
    img_bytes = resp.read()

arr = np.frombuffer(img_bytes, np.uint8)
img_bgr = cv2.imdecode(arr, cv2.IMREAD_COLOR)
if img_bgr is None:
    raise ValueError(f"Could not decode image from URL: {IMAGE_URL}")

img_rgb_orig = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)   # keep original for display
img_resized  = cv2.resize(img_rgb_orig, (256, 256))        # model input size

# ── 2. Pre-process (same normalisation used during training) ──────────────────
infer_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
tensor = infer_transform(image=img_resized)["image"].unsqueeze(0)  # (1,3,256,256)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor = tensor.to(device)

# ── 3. Load best checkpoints (reuse callbacks defined in training cells) ───────
unet_ckpt_path = unet_ckpt_cb.best_model_path
transunet_ckpt_path = transunet_ckpt_cb.best_model_path

if not unet_ckpt_path or not os.path.isfile(unet_ckpt_path):
    raise FileNotFoundError(
        "U-Net checkpoint not found. Please ensure the training cell `4f42948f` "
        "ran successfully and saved a best model checkpoint."
    )

if not transunet_ckpt_path or not os.path.isfile(transunet_ckpt_path):
    raise FileNotFoundError(
        "TransUNet checkpoint not found. Please ensure the training cell `4f42948f` "
        "ran successfully and saved a best model checkpoint."
    )

unet_infer      = UNetSystem.load_from_checkpoint(
    unet_ckpt_path, in_channels=3, out_channels=1, lr=1e-4
).eval().to(device)

transunet_infer = TransUNetSystem.load_from_checkpoint(
    transunet_ckpt_path, in_channels=3, out_channels=1, lr=1e-4
).eval().to(device)

# ── 4. Inference ──────────────────────────────────────────────────────────────
with torch.no_grad():
    unet_mask      = (torch.sigmoid(unet_infer(tensor))      >= THRESHOLD).squeeze().cpu().numpy()  # bool (256,256)
    transunet_mask = (torch.sigmoid(transunet_infer(tensor)) >= THRESHOLD).squeeze().cpu().numpy()

# ── 5. Helper: colour overlay ─────────────────────────────────────────────────
def overlay(base_rgb: np.ndarray, mask: np.ndarray, colour: tuple, alpha: float) -> np.ndarray:
    """Blend a semi-transparent coloured mask onto a copy of base_rgb."""
    out = base_rgb.copy().astype(np.float32)
    colour_layer = np.zeros_like(out)
    colour_layer[mask == 1] = colour
    out = (1 - alpha) * out + alpha * colour_layer
    out = np.clip(out, 0, 255).astype(np.uint8)
    # Draw a thin contour for crisp boundary
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, colour, 2)
    return out

unet_vis      = overlay(img_resized, unet_mask,      UNET_COLOR,      ALPHA)
transunet_vis = overlay(img_resized, transunet_mask, TRANSUNET_COLOR, ALPHA)

# ── 6. Plot ───────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))

panels = [
    (img_resized,    "Original (256×256)",                  None),
    (unet_vis,       "U-Net prediction overlay",            UNET_COLOR),
    (transunet_vis,  "TransUNet prediction overlay",        TRANSUNET_COLOR),
]

for ax, (img, title, colour) in zip(axes, panels):
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold",
                 color=("#%02x%02x%02x" % colour) if colour else "black")
    ax.axis("off")
    if colour:
        # Coverage stats in subtitle
        mask = unet_mask if colour == UNET_COLOR else transunet_mask
        pct  = 100 * mask.mean()
        ax.set_xlabel(f"Lesion coverage: {pct:.1f}%", fontsize=10)

fig.suptitle(f"Segmentation Inference\n{IMAGE_URL}", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f"U-Net      lesion coverage : {100*unet_mask.mean():.2f}%")
print(f"TransUNet  lesion coverage : {100*transunet_mask.mean():.2f}%")
